<a href="https://colab.research.google.com/github/Sulamithsingh/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Sulamithsingh/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:

relevant_columns = [
    "content_id",
    "client_id",
    "position_tier",
    "avg_position",
    "clicks_last_30d",
    "clicks_prev_30d",
    "impressions_90d",
    "ctr",
    "trend_pct",
    "trend_direction"
]

display(df[relevant_columns].head())

,content_id,client_id,position_tier,avg_position,clicks_last_30d,clicks_prev_30d,impressions_90d,ctr,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,striking,10.6,2,13,3803,0.76,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,20.3,2,1,15320,0.05,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,36.5,1,3,12581,0.09,-60.9,down
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,22,17,11751,0.49,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,44.0,10,2,19140,0.13,-34.7,down


In [3]:

priority_segment = df[
    (df["position_tier"] == "striking") &
    (df["clicks_last_30d"] > 0) &
    (df["trend_direction"] == "down")
].copy()

print("Priority content items:", len(priority_segment))
print("Average position:", round(priority_segment["avg_position"].mean(), 1))
print("Median trend (%):", round(priority_segment["trend_pct"].median(), 1))
print("Clicks - last 30 days:", priority_segment["clicks_last_30d"].sum())
print("Clicks - previous 30 days:", priority_segment["clicks_prev_30d"].sum())

Priority content items: 1561
Average position: 14.3
Median trend (%): -44.4
Clicks - last 30 days: 10171
Clicks - previous 30 days: 13642


In [6]:


action_queue = pd.DataFrame([
    {
        "priority": 1,
        "reason_code": "STRIKING_DISTANCE_DECLINE",
        "action": "Prioritize human review for striking-distance pages with declining clicks",
        "evidence": "1,561 pages have recent clicks, are in striking distance, and show a downward trend. Median trend is -44.4%.",
        "recommended_review": "Review factual freshness, content completeness, internal linking, and page/query relevance."
    },
    {
        "priority": 2,
        "reason_code": "NO_CLICKS_WITH_IMPRESSIONS",
        "action": "Investigate pages receiving impressions but no recent clicks",
        "evidence": "18,365 pages have impressions but zero clicks in the last 30 days.",
        "recommended_review": "Check indexing, search visibility, title/snippet presentation, ranking position, and page/query relevance before making content changes."
    },
    {
        "priority": 3,
        "reason_code": "STRIKING_DISTANCE_STABLE",
        "action": "Monitor stable striking-distance pages before making major changes",
        "evidence": "823 pages have recent clicks and are in striking distance, with a median trend of -5.4%.",
        "recommended_review": "Monitor clicks, impressions, CTR, and average position in the next reporting period."
    },
    {
        "priority": 4,
        "reason_code": "STRIKING_DISTANCE_IMPROVING",
        "action": "Continue monitoring improving striking-distance pages",
        "evidence": "489 pages have recent clicks and are improving, with a median trend of +54.4%.",
        "recommended_review": "Avoid unnecessary changes and monitor whether the improvement persists."
    }
])

display(action_queue)

,priority,reason_code,action,evidence,recommended_review
0,1,STRIKING_DISTANCE_DECLINE,Prioritize human review for striking-distance ...,"1,561 pages have recent clicks, are in strikin...","Review factual freshness, content completeness..."
1,2,NO_CLICKS_WITH_IMPRESSIONS,Investigate pages receiving impressions but no...,"18,365 pages have impressions but zero clicks ...","Check indexing, search visibility, title/snipp..."
2,3,STRIKING_DISTANCE_STABLE,Monitor stable striking-distance pages before ...,823 pages have recent clicks and are in striki...,"Monitor clicks, impressions, CTR, and average ..."
3,4,STRIKING_DISTANCE_IMPROVING,Continue monitoring improving striking-distanc...,489 pages have recent clicks and are improving...,Avoid unnecessary changes and monitor whether ...


In [5]:


segments = {
    "Striking distance + declining": (
        (df["position_tier"] == "striking") &
        (df["clicks_last_30d"] > 0) &
        (df["trend_direction"] == "down")
    ),

    "Striking distance + stable": (
        (df["position_tier"] == "striking") &
        (df["clicks_last_30d"] > 0) &
        (df["trend_direction"] == "stable")
    ),

    "Striking distance + improving": (
        (df["position_tier"] == "striking") &
        (df["clicks_last_30d"] > 0) &
        (df["trend_direction"] == "up")
    ),

    "No clicks + impressions": (
        (df["clicks_last_30d"] == 0) &
        (df["impressions_90d"] > 0)
    )
}

segment_results = []

for name, condition in segments.items():
    subset = df[condition]

    segment_results.append({
        "segment": name,
        "items": len(subset),
        "total_clicks_30d": subset["clicks_last_30d"].sum(),
        "median_impressions_90d": subset["impressions_90d"].median(),
        "median_avg_position": subset["avg_position"].median(),
        "median_trend_pct": subset["trend_pct"].median()
    })

segment_summary = pd.DataFrame(segment_results)

display(segment_summary)

,segment,items,total_clicks_30d,median_impressions_90d,median_avg_position,median_trend_pct
0,Striking distance + declining,1561,10171,3562.0,14.0,-44.4
1,Striking distance + stable,823,8212,3450.0,13.6,-5.4
2,Striking distance + improving,489,5753,2500.0,13.5,54.4
3,No clicks + impressions,18365,0,175.0,12.1,-44.8


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This action playbook is intended to support human review and prioritization of content actions using observed performance patterns in the FlyRank starter dataset.

It can be used to:
- Prioritize pages for content review based on clicks, impressions, ranking position, and observed trend direction.
- Identify segments that may deserve closer investigation.
- Support consistent content-refresh and monitoring decisions.
- Provide a repeatable decision-support framework for future reporting periods.

### Limits

The recommendations are decision-support signals, not predictions or guarantees of future performance.

The analysis is observational, so the measured relationships do not establish that a specific content change will cause an improvement in clicks or rankings.

The dataset does not contain query text, conversion/revenue data, or all technical SEO and SERP factors. Therefore, this playbook should not be used to infer search intent, business impact, or the cause of a performance change without additional evidence.

All recommended actions should be reviewed by a human before implementation.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review before action

Every recommended action must be reviewed by a human before implementation.

The reviewer should:
- Confirm that the page belongs to the identified performance segment.
- Check the underlying clicks, impressions, CTR, and ranking signals.
- Review the page itself for factual freshness, completeness, and relevance.
- Consider technical SEO, indexing, SERP changes, seasonality, and other external factors.
- Record the reason for the final action and any relevant evidence.

### No-go list

The playbook must not be used to:

- Automatically publish, delete, or rewrite content without human approval.
- Claim that a content change will definitely improve rankings or traffic.
- Infer user search intent from fields that do not contain query information.
- Make claims about revenue, conversions, or business impact when those fields are not present.
- Treat correlation or observed trends as proof of causation.
- Treat `avg_position = 0` as a real ranking position.
- Use pseudonymized `content_id` or `client_id` as predictive features.
- Make recommendations based on private queries, client identities, or other non-public information.
- Override technical, editorial, legal, or brand review requirements.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring plan

The action playbook should be monitored over each reporting period using the same metrics and segment definitions.

Track:
- Number of pages in each priority segment.
- Clicks in the last 30 days compared with the previous 30 days.
- Impressions and CTR.
- Average position, treating `avg_position = 0` as missing/no-data rather than a valid rank.
- Trend direction and trend percentage.
- Changes in the distribution of pages across the priority segments.

### Review and retrain triggers

A review should be triggered when:
- Segment sizes change substantially from the previous reporting period.
- The direction or magnitude of observed trends changes materially.
- Important input fields become missing, inconsistent, or change definition.
- The dataset schema changes or new features are introduced.
- The relationship between the measured signals and the resulting action priorities changes enough to make the existing rules unreliable.

If an ML model is used in a future version, retraining should only be considered after checking for data drift, target-definition changes, leakage, and client or time-based distribution changes.

Model performance should be evaluated on a leakage-safe validation strategy rather than relying only on random-split accuracy.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
# Create paper-ready exports

action_queue_export = action_queue.copy()
segment_summary_export = segment_summary.copy()

# Save clean CSV files
action_queue_export.to_csv("action_queue_for_paper.csv", index=False)
segment_summary_export.to_csv("segment_summary_for_paper.csv", index=False)

print("Exports created:")
print("- action_queue_for_paper.csv")
print("- segment_summary_for_paper.csv")

print("\nAction queue:")
display(action_queue_export)

print("\nSegment summary:")
display(segment_summary_export)

Exports created:
- action_queue_for_paper.csv
- segment_summary_for_paper.csv

Action queue:


,priority,reason_code,action,evidence,recommended_review
0,1,STRIKING_DISTANCE_DECLINE,Prioritize human review for striking-distance ...,"1,561 pages have recent clicks, are in strikin...","Review factual freshness, content completeness..."
1,2,NO_CLICKS_WITH_IMPRESSIONS,Investigate pages receiving impressions but no...,"18,365 pages have impressions but zero clicks ...","Check indexing, search visibility, title/snipp..."
2,3,STRIKING_DISTANCE_STABLE,Monitor stable striking-distance pages before ...,823 pages have recent clicks and are in striki...,"Monitor clicks, impressions, CTR, and average ..."
3,4,STRIKING_DISTANCE_IMPROVING,Continue monitoring improving striking-distanc...,489 pages have recent clicks and are improving...,Avoid unnecessary changes and monitor whether ...



Segment summary:


,segment,items,total_clicks_30d,median_impressions_90d,median_avg_position,median_trend_pct
0,Striking distance + declining,1561,10171,3562.0,14.0,-44.4
1,Striking distance + stable,823,8212,3450.0,13.6,-5.4
2,Striking distance + improving,489,5753,2500.0,13.5,54.4
3,No clicks + impressions,18365,0,175.0,12.1,-44.8


### Paper-ready takeaway

The strongest observed prioritization signal is the striking-distance segment with declining clicks. This segment contains 1,561 pages, with a median trend of -44.4% and 10,171 clicks during the last 30 days. These pages should receive priority for human content review.

A second investigation group contains 18,365 pages with impressions but zero clicks in the last 30 days. This group is large, but the available data does not establish why these pages receive no clicks, so technical, ranking, and page-level factors should be reviewed before recommending content changes.

Stable and improving striking-distance pages should primarily be monitored rather than automatically changed.

These findings are observational and should be treated as decision-support signals rather than causal or predictive claims.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.